# Rich dump run (Test_Iteration_6)

Runs **all 27 wide queries** (headings -10..+10 step 2.5 x scales 0.95, 1.0, 1.05) on every photo and saves, per photo, every candidate of every query
**with its raw LoFTR matches**, the photo's attitude and each query's seconds per stage (block `train/checks/rich_dump.py`).

Why: the older dumps (Test_Iteration_3) keep only inliers and shifts. With the raw matches every later idea can be tested offline, without the GPU:
the decision (bar, agreement, escalation, learned verifier), the fit itself (RANSAC variants, affine, homography, confidence cut), the position
(roll/pitch bias) and the query policy (which queries, in which order, when to stop: the Jetson time budget). The standard 6 queries are a subset
of the 27, so today's pipeline (agreement + escalation) is replayed exactly from the same file.

Settings by environment: `GATE_DATA` (official / nudt), `NUDT_SEED`, `N_NUDT`, `EXCLUDE_SEEDS` (comma list: skip the photos those seeds picked, for a
fresh sample), `N_LIMIT` (only the first n photos, for a smoke test), `TAG`.

In [1]:
import os, sys, time
from pathlib import Path
import numpy as np
import pandas as pd

ROOT = Path.cwd()
while not (ROOT / "single_photo_pipeline.py").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT)); sys.path.insert(0, str(ROOT / "src")); os.chdir(ROOT)

DATA = os.environ.get("GATE_DATA", "official")
MAP = "new"
POOL_PHOTOS = int(os.environ.get("POOL_PHOTOS", "500"))   # same pool and setup seed as Test_Iteration_3, so NUDT seed 101 / 202 pick the same photos
WIDE_HEADINGS = [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0]
WIDE_SCALES = [0.95, 1.0, 1.05]
K_PER_QUERY, M_PER_QUERY = 50, 10
MATCHER_CONFIDENCE, LOFTR_COARSE_THRESHOLD = 0.22, 0.21
MIN_INLIERS, GAMMA = 8, 1.0
NUDT_SEED = int(os.environ.get("NUDT_SEED", "101"))
N_NUDT = int(os.environ.get("N_NUDT", "300"))
EXCLUDE_SEEDS = [int(s) for s in os.environ.get("EXCLUDE_SEEDS", "").split(",") if s.strip()]
N_LIMIT = int(os.environ.get("N_LIMIT", "0"))
EXCLUDE_N = int(os.environ.get("EXCLUDE_N", "300"))   # size of the excluded seeds' samples (seed 101 picked 300; this reproduces Test_Iteration_3's seed 202 photos exactly)
MATCHER = os.environ.get("MATCHER", "loftr")          # "loftr" (the pipeline) or "xoftr" (MINIMA XoFTR, a swap-in with the same plumbing)
XOFTR_COARSE, XOFTR_FINE = float(os.environ.get("XOFTR_COARSE", "0.3")), float(os.environ.get("XOFTR_FINE", "0.1"))
FRESH_EXCLUDE_POOL = int(os.environ.get("FRESH_EXCLUDE_POOL", "0"))   # >0 (NUDT, with POOL_PHOTOS >= all frames): skip every photo of the usual
# pool of this size (setup seed 7), so the sample is from photos no earlier test has used
EXCLUDE_IDS_FILE = os.environ.get("EXCLUDE_IDS_FILE")   # optional csv with a photo column: never draw these photos (an earlier fresh sample)
PHOTOS_FILE = os.environ.get("PHOTOS_FILE")           # optional csv with a photo column: only these photos
print(MATCHER, DATA, "seed", NUDT_SEED, "n", N_NUDT, "exclude", EXCLUDE_SEEDS, "limit", N_LIMIT)

loftr nudt seed 1212 n 150 exclude [] limit 0


In [2]:
# NUDT VALID split (other flights than the test split every earlier draw used): the pipeline's setup loads the test split, so swap the loader here
import functools
from train.datasets import photo_sets
photo_sets.load_nudt_photos = functools.partial(photo_sets.load_nudt_photos, split=os.environ.get("NUDT_SPLIT", "test"))
from train.checks.rich_dump import RichDumpPipeline, pack_photo
pipeline = RichDumpPipeline(DATA, "edge_zncc", semantic_layer=True, semantic_decision=True, gamma=GAMMA, k_peaks=K_PER_QUERY, m_fine=M_PER_QUERY,
                            matcher="loftr_minima", min_inliers=MIN_INLIERS, ransac_pixels=None, matcher_confidence=MATCHER_CONFIDENCE, satellite=MAP,
                            map_margin_m=None, loftr_coarse_threshold=LOFTR_COARSE_THRESHOLD, dino_weight=None,
                            headings=WIDE_HEADINGS, scale_factors=WIDE_SCALES)
start = time.time()
pipeline.setup(POOL_PHOTOS, 7)
print("setup done in %.1f s" % (time.time() - start))
for line in pipeline.describe():
    print(line)
test_set = pipeline.test_set
if MATCHER == "xoftr":
    from train.checks.xoftr_batched import load_xoftr_batched      # the src loader lacks the batch ids the batched LoFTR plumbing reads
    pipeline.matcher = load_xoftr_batched(ROOT / "dataset/NUDT/checkpoint/MINIMA/minima_xoftr.ckpt", min_confidence=XOFTR_FINE,
                                          coarse_threshold=XOFTR_COARSE, fine_threshold=XOFTR_FINE)
    print("matcher: XoFTR (coarse %.2f, fine %.2f)" % (XOFTR_COARSE, XOFTR_FINE))

level 3: 0.85 m/pixel, picture 14481 x 9818, map coverage 96%
level 4: 1.20 m/pixel, picture 10240 x 6942, map coverage 96%


map store ready, took 25 s


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/kornia/feature/lightglue.py:44: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd(cast_inputs=torch.float32)


/home/voidness/Storage/Synced Folders/InterIIT Tech 15/venv-geoloc/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


setup done in 34.3 s
data: nudt | 1425 photos available | zoom levels [3, 4]
photos: nudt | map: NUDT reference map
coarse method: edge_zncc | maps: ['edge_zncc']
semantic layer: on (best K=50 -> M=10) | DINO in the decision: on (gamma 1.00, rescue (None, None))
fine matcher: loftr_minima | answer bar: 8 inliers | RANSAC 3.0 px | matcher confidence 0.22 | LoFTR coarse threshold 0.21, upscale 1.0 | border 0 px, margin 0 px
DINO order weight: 100.0 | projection overrides: none (settings.GEOMETRY)
batches: 1 LoFTR pairs and 25 DINO places per call
device: cuda
queries per photo: 27 (headings [-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0], scale factors [0.95, 1.0, 1.05]); K and M are per query; 1 queries matched together


In [3]:
n_all = len(test_set.items)
if DATA == "official":
    indices = list(range(n_all)); tag = "official"
else:
    pool = np.arange(n_all)
    seen = set()
    for s in EXCLUDE_SEEDS:              # same draw as Decision_Dump_Run: default_rng(seed).choice(n_all, N_NUDT) on the whole pool
        seen |= set(np.random.default_rng(s).choice(n_all, EXCLUDE_N, replace=False).tolist())
    if FRESH_EXCLUDE_POOL:
        from train.datasets import nudt_data
        assert len(test_set.items) > FRESH_EXCLUDE_POOL, "POOL_PHOTOS must load more photos than the excluded pool"
        seen |= set(int(i) for i in nudt_data.spread_sample(test_set.items, FRESH_EXCLUDE_POOL, 7))
    if EXCLUDE_IDS_FILE:
        seen |= set(int(i) for i in pd.read_csv(EXCLUDE_IDS_FILE).photo)
    pool = np.array([i for i in pool if i not in seen])
    indices = sorted(int(i) for i in np.random.default_rng(NUDT_SEED).choice(pool, min(N_NUDT, len(pool)), replace=False))
    tag = ("nudt_fresh%d" if FRESH_EXCLUDE_POOL else "nudt_seed%d") % NUDT_SEED
if PHOTOS_FILE:
    indices = sorted(int(i) for i in pd.read_csv(PHOTOS_FILE).photo)
if N_LIMIT:
    indices = indices[:N_LIMIT]
tag = os.environ.get("TAG", tag + ("" if MATCHER == "loftr" else "_" + MATCHER))
DUMP = ROOT / "tests/Test_Iteration_7/results/rich_dump" / tag
DUMP.mkdir(parents=True, exist_ok=True)
done = {int(p.stem.split("_")[1]) for p in DUMP.glob("photo_*.npz")}
print(tag, len(indices), "photos,", len(done & set(indices)), "already done ->", DUMP)

nudt_valid1212 150 photos, 0 already done -> /home/voidness/Storage/Synced Folders/InterIIT Tech 15/IdeaForge-GeoLoc-InterIIT15/tests/Test_Iteration_7/results/rich_dump/nudt_valid1212


In [4]:
import torch
start = time.time(); failures = []; sizes = []
todo = [i for i in indices if i not in done]
for count, index in enumerate(todo):
    item = test_set.items[index]
    try:
        found = None
        for attempt in range(3):
            try:
                found = pipeline.run_queries(item, test_set.priors[index]); break
            except torch.cuda.OutOfMemoryError:
                torch.cuda.empty_cache(); time.sleep(5)
        if found is None:
            raise MemoryError("GPU out of memory three times")
        arrays = pack_photo(item, test_set.truths[index], test_set.sigmas[index], found["level"], found["queries"],
                            min_keep_confidence=min(0.15, XOFTR_FINE) if MATCHER == "xoftr" else 0.15)
        path = DUMP / ("photo_%d.npz" % index)
        np.savez_compressed(path, **arrays)
        sizes.append(path.stat().st_size)
    except Exception as error:
        failures.append((index, repr(error)[:200])); print("photo", index, "failed", repr(error)[:200])
    if count % 10 == 0:
        print("photo %d/%d | %.0f s | %.1f s per photo | %.0f kB per file" % (count, len(todo), time.time() - start,
              (time.time() - start) / (count + 1), np.mean(sizes) / 1e3 if sizes else 0), flush=True)
print("done", len(todo), "photos in %.0f s;" % (time.time() - start), len(failures), "failed")
pd.DataFrame(failures, columns=["photo", "error"]).to_csv(DUMP / "failures.csv", index=False)

photo 0/150 | 20 s | 19.6 s per photo | 119 kB per file


photo 10/150 | 220 s | 20.0 s per photo | 124 kB per file


photo 20/150 | 422 s | 20.1 s per photo | 130 kB per file


photo 30/150 | 612 s | 19.8 s per photo | 130 kB per file


photo 40/150 | 811 s | 19.8 s per photo | 129 kB per file


photo 50/150 | 1008 s | 19.8 s per photo | 127 kB per file


... (6 outputs left out)

photo 120/150 | 2343 s | 19.4 s per photo | 128 kB per file


photo 130/150 | 2531 s | 19.3 s per photo | 128 kB per file


photo 140/150 | 2718 s | 19.3 s per photo | 127 kB per file


done 150 photos in 2889 s; 0 failed


## Check: the stored matches give back the pipeline's inliers

Re-fit every candidate of the first photos from the decoded matches with the pipeline's own `stage2_from_matches` and compare with the stored
inlier counts. A mismatch would mean the packing loses information.

In [5]:
import cv2
from train.checks.rich_dump import load_rich
from fine_matching.loftr_matcher import stage2_from_matches
from header.settings import SEED
same = total = 0
for path in sorted(DUMP.glob("photo_*.npz"))[:10]:
    photo = load_rich(path)
    for query in photo["queries"]:
        stored = dict(inside=query["inside"], margin=query["margin"], border_px=query["border_px"], items=query["matches"])
        cv2.setRNGSeed(SEED)
        again = stage2_from_matches(stored, photo["level"], MATCHER_CONFIDENCE)
        same += int((again["inliers"] == query["inliers"]).sum()); total += len(query["inliers"])
print("candidates with the same inliers after decoding: %d of %d" % (same, total))

candidates with the same inliers after decoding: 2688 of 2700
